# Sensitivity analysis: recommender scoring modes

Ten fixed personas, three `scoring_mode` values (`similarity`, `weighted_average`, `cosine`). Baseline: top 3 metros per persona per mode. Then each dimension slider is nudged by plus or minus 0.5 (clipped to 0 through 5), one dimension at a time; again top 3 saved. Use the tables and plots below to compare modes and how stable rankings are to small preference tweaks.

**Run from project root** so `data/final/Final_Enriched_Dataset.csv` loads.

## Steps: setup

1. Put the project root on `sys.path`.
2. Import libraries and `recommend_cities`, `DIMENSION_SCORE_COLS`, `SCORE_MAX`.
3. Load `data/final/Final_Enriched_Dataset.csv` with `cbsa_code` as string.

In [ ]:
# 1. project root on path
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 2. imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.recommender import recommend_cities, DIMENSION_SCORE_COLS, SCORE_MAX

# 3. load enriched panel (0-1 scores; recommender returns 0-5 for display)
df = pd.read_csv(
    "data/final/Final_Enriched_Dataset.csv",
    dtype={"cbsa_code": str},
)


## Steps: constants and personas

1. Set `SCORING_MODES`, `TOP_N`, `DELTA`.
2. Define `PERSONAS`: each has `persona_id`, `label`, and `prefs` (all keys in `DIMENSION_SCORE_COLS`, values 0-5).

In [ ]:
# 1. run constants
SCORING_MODES = ("similarity", "weighted_average", "cosine")
TOP_N = 3
DELTA = 0.5

# 2. ten personas — values listed in DIMENSION_SCORE_COLS order
ORDER = DIMENSION_SCORE_COLS


def P(vals):
    return dict(zip(ORDER, vals))


PERSONAS = [
    {"persona_id": 1, "label": "Balanced", "prefs": P([2.5] * 10)},
    {"persona_id": 2, "label": "Affordability_first", "prefs": P([5, 2, 2, 2, 2, 2, 2, 2, 2, 2])},
    {"persona_id": 3, "label": "Jobs_urban", "prefs": P([2, 5, 2, 2, 2, 2, 2, 5, 2, 2])},
    {"persona_id": 4, "label": "Safety_education", "prefs": P([2, 2, 5, 5, 2, 2, 2, 2, 2, 2])},
    {"persona_id": 5, "label": "Health_walkability", "prefs": P([2, 2, 2, 2, 5, 5, 2, 2, 2, 2])},
    {"persona_id": 6, "label": "Weather", "prefs": P([2, 2, 2, 2, 2, 2, 2, 2, 5, 5])},
    {"persona_id": 7, "label": "Diversity_education", "prefs": P([2, 2, 2, 4, 2, 2, 5, 2, 2, 2])},
    {"persona_id": 8, "label": "Suburban", "prefs": P([4, 2, 4, 4, 2, 2, 2, 1, 2, 2])},
    {"persona_id": 9, "label": "Sun_belt", "prefs": P([3, 2, 2, 2, 2, 2, 2, 2, 5, 4])},
    {"persona_id": 10, "label": "Mixed_priorities", "prefs": P([3, 4, 3, 3, 3, 4, 3, 2, 3, 2])},
]


## Steps: helpers

1. `perturb(prefs, dim, delta)` — copy prefs, add delta to one dimension, clip to 0 through `SCORE_MAX`.
2. `jaccard_top3(a, b)` — Jaccard similarity of two top-3 CBSA code lists treated as sets.

In [ ]:
# 1. one-slider nudge
def perturb(prefs, dim, delta):
    out = dict(prefs)
    out[dim] = float(np.clip(out[dim] + delta, 0.0, SCORE_MAX))
    return out


# 2. overlap of top-3 CBSA sets
def jaccard_top3(a, b):
    sa, sb = set(a), set(b)
    return len(sa & sb) / len(sa | sb)


## Steps: baseline

1. For each persona and each scoring mode, call `recommend_cities` with `top_n=3`, no income filter.
2. Record rank 1 through 3, metro fields, and `recommendation_score` into `baseline_top3`.

In [ ]:
# 1. run baseline grid
rows = []
for p in PERSONAS:
    for mode in SCORING_MODES:
        top = recommend_cities(
            df,
            user_inputs=p["prefs"],
            user_income=None,
            housing_mode="either",
            top_n=TOP_N,
            scoring_mode=mode,
        )
        # 2. flatten top 3 with rank
        for rank, (_, row) in enumerate(top.iterrows(), start=1):
            rows.append(
                {
                    "persona_id": p["persona_id"],
                    "persona_label": p["label"],
                    "scoring_mode": mode,
                    "rank": rank,
                    "cbsa_code": row["cbsa_code"],
                    "cbsa_name": row["cbsa_name"],
                    "city": row["city"],
                    "state": row["state"],
                    "recommendation_score": row["recommendation_score"],
                }
            )

baseline_top3 = pd.DataFrame(rows)
display(baseline_top3.head(12))


## Steps: perturbations

1. For each persona, each dimension in `DIMENSION_SCORE_COLS`, each delta in minus and plus `DELTA`, each scoring mode, build perturbed prefs and run `recommend_cities` with `top_n=3`.
2. Append rows to `perturbation_top3` (include `perturbed_dimension`, `delta`).

In [ ]:
# 1. perturbation grid
rows = []
for p in PERSONAS:
    base = p["prefs"]
    for dim in DIMENSION_SCORE_COLS:
        for delta in (-DELTA, DELTA):
            prefs_p = perturb(base, dim, delta)
            for mode in SCORING_MODES:
                top = recommend_cities(
                    df,
                    user_inputs=prefs_p,
                    user_income=None,
                    housing_mode="either",
                    top_n=TOP_N,
                    scoring_mode=mode,
                )
                for rank, (_, row) in enumerate(top.iterrows(), start=1):
                    rows.append(
                        {
                            "persona_id": p["persona_id"],
                            "persona_label": p["label"],
                            "perturbed_dimension": dim,
                            "delta": delta,
                            "scoring_mode": mode,
                            "rank": rank,
                            "cbsa_code": row["cbsa_code"],
                            "cbsa_name": row["cbsa_name"],
                            "city": row["city"],
                            "state": row["state"],
                            "recommendation_score": row["recommendation_score"],
                        }
                    )

perturbation_top3 = pd.DataFrame(rows)
display(perturbation_top3.head(12))


## Steps: save (optional)

1. Uncomment the `to_csv` lines to write CSVs next to this notebook under `exploratory_notebooks/`.

In [ ]:
# 1. optional: persist tables (uncomment to run)
# baseline_top3.to_csv("exploratory_notebooks/sensitivity_baseline_top3.csv", index=False)
# perturbation_top3.to_csv("exploratory_notebooks/sensitivity_perturbation_top3.csv", index=False)


## Steps: stability summary and plots

1. Build baseline top-3 sets per `(persona_id, scoring_mode)`.
2. For each perturbation run, compute Jaccard vs baseline; aggregate mean Jaccard into `stability_summary` (persona by mode).
3. Draw heatmaps and bar charts.

In [ ]:
# 1. baseline top-3 CBSA sets
baseline_sets = baseline_top3.groupby(["persona_id", "scoring_mode"])["cbsa_code"].apply(list).to_dict()

# 2. Jaccard for each perturbation scenario
jac_rows = []
for key, g in perturbation_top3.groupby(
    ["persona_id", "perturbed_dimension", "delta", "scoring_mode"]
):
    pid, dim, delta, mode = key
    pert_codes = g.sort_values("rank")["cbsa_code"].tolist()
    base_codes = baseline_sets[(pid, mode)]
    jac_rows.append(
        {
            "persona_id": pid,
            "perturbed_dimension": dim,
            "delta": delta,
            "scoring_mode": mode,
            "jaccard": jaccard_top3(base_codes, pert_codes),
            "exact_top3_match": set(base_codes) == set(pert_codes),
        }
    )

jaccard_runs = pd.DataFrame(jac_rows)

stability_summary = jaccard_runs.groupby(["persona_id", "scoring_mode"])["jaccard"].mean().unstack("scoring_mode")
display(stability_summary)

exact_rate = jaccard_runs.groupby(["persona_id", "scoring_mode"])["exact_top3_match"].mean().unstack("scoring_mode")
display(exact_rate)


In [ ]:
# 1. plot style
%matplotlib inline
sns.set_theme(style="whitegrid")

# 2. stability heatmap (mean Jaccard vs baseline top 3)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(stability_summary, annot=True, fmt=".2f", cmap="viridis", ax=ax)
ax.set_title("Mean Jaccard: baseline top 3 vs perturbed top 3")
plt.tight_layout()
plt.show()

# 3. baseline: top-1 recommendation_score by persona and mode
top1 = baseline_top3[baseline_top3["rank"] == 1].copy()
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=top1, x="persona_id", y="recommendation_score", hue="scoring_mode", ax=ax)
ax.set_title("Baseline top-1 recommendation_score on 0-5 scale")
plt.tight_layout()
plt.show()

# 4. sensitivity heatmap: mean Jaccard by perturbed dimension and persona (one facet per mode)
for mode in SCORING_MODES:
    sub = jaccard_runs[jaccard_runs["scoring_mode"] == mode]
    heat = sub.groupby(["persona_id", "perturbed_dimension"])["jaccard"].mean().unstack("perturbed_dimension")
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.heatmap(heat, cmap="mako", ax=ax)
    ax.set_title(f"Sensitivity: mean Jaccard vs baseline, mode={mode}")
    plt.tight_layout()
    plt.show()

# 5. cross-mode agreement: distinct CBSAs in union of baseline top-3 across modes
union_rows = []
for pid in sorted(baseline_top3["persona_id"].unique()):
    codes = set()
    for mode in SCORING_MODES:
        codes.update(baseline_sets[(pid, mode)])
    union_rows.append({"persona_id": pid, "n_distinct_in_union": len(codes)})
union_df = pd.DataFrame(union_rows)
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=union_df, x="persona_id", y="n_distinct_in_union", ax=ax)
ax.set_title("Distinct CBSAs in union of baseline top-3 across all three modes (3 means full agreement)")
plt.tight_layout()
plt.show()

# 6. pairwise Jaccard between modes on baseline top-3 (per persona)
pair_rows = []
modes_list = list(SCORING_MODES)
for pid in sorted(baseline_top3["persona_id"].unique()):
    for i, m1 in enumerate(modes_list):
        for m2 in modes_list[i + 1 :]:
            a = baseline_sets[(pid, m1)]
            b = baseline_sets[(pid, m2)]
            pair_rows.append(
                {
                    "persona_id": pid,
                    "mode_a": m1,
                    "mode_b": m2,
                    "jaccard": jaccard_top3(a, b),
                }
            )
pair_df = pd.DataFrame(pair_rows)
display(pair_df.pivot_table(index="persona_id", columns=["mode_a", "mode_b"], values="jaccard"))

# 7. optional: metros that show up most often in perturbed top-3 for cosine
mode_pick = "cosine"
subp = perturbation_top3[perturbation_top3["scoring_mode"] == mode_pick]
vc = subp["cbsa_code"].value_counts().head(15)
fig, ax = plt.subplots(figsize=(8, 4))
vc.plot.bar(ax=ax, color="steelblue")
ax.set_title(f"Top CBSAs by count in perturbed top-3, mode={mode_pick}")
plt.tight_layout()
plt.show()


## Your notes

- Which `scoring_mode` fits each persona best (face validity)?
- Which mode is most stable under plus or minus 0.5 one-dimension nudges?
- Any personas where the three modes disagree strongly (see union and pairwise Jaccard)?

---

- 
- 
- 